# Himalayan Climbing Expeditions (1905-2019)

Cleaning and transformation

## Reading the raw data

In [1]:
import pandas as pd

raw_peaks = pd.read_csv("../data/raw/peaks.csv")
raw_expeditions = pd.read_csv("../data/raw/expeditions.csv")
raw_members = pd.read_csv("../data/raw/members.csv")
print(raw_peaks.shape, raw_expeditions.shape, raw_members.shape)

(468, 8) (10364, 16) (76519, 21)


## Data cleaning

Cleaning is based on the problems found in the EDA. We start again from the raw copies so the EDA columns we added are not carried over.

In [2]:
peaks = raw_peaks.copy()
expeditions = raw_expeditions.copy()
members = raw_members.copy()

### Duplicate ids

In [3]:
print(expeditions["expedition_id"].duplicated().sum(), "duplicate expedition_id")
print(members["member_id"].duplicated().sum(), "duplicate member_id")
expeditions[expeditions["expedition_id"] == "KANG10101"][["expedition_id", "peak_name", "year"]]

1 duplicate expedition_id
1 duplicate member_id


,expedition_id,peak_name,year
2883,KANG10101,Kangchenjunga,1910
6819,KANG10101,Kangchenjunga,2010


In [4]:
print("KANG10100" in set(expeditions["expedition_id"]))

old = (expeditions["expedition_id"] == "KANG10101") & (expeditions["year"] == 1910)
expeditions.loc[old, "expedition_id"] = "KANG10100"

old_m = (members["expedition_id"] == "KANG10101") & (members["year"] == 1910)
members.loc[old_m, "expedition_id"] = "KANG10100"
members.loc[old_m, "member_id"] = members.loc[old_m, "member_id"].str.replace("KANG10101", "KANG10100")

print(expeditions["expedition_id"].duplicated().sum(), members["member_id"].duplicated().sum())

False
0 0


### Peak missing from the peaks table

Peak `SPHU` is used in expeditions and members but is not in `peaks`. We add it to `peaks` with the information we have so the tables can be linked.

In [5]:
missing_peaks = set(expeditions["peak_id"]) - set(peaks["peak_id"])
print(missing_peaks)

new_peak = pd.DataFrame({"peak_id": ["SPHU"], "peak_name": ["Unknown peak (SPHU)"], "climbing_status": ["Unknown"]})
peaks = pd.concat([peaks, new_peak], ignore_index=True)
print(set(expeditions["peak_id"]) - set(peaks["peak_id"]))

{'SPHU'}
set()


### Fix dates

In [6]:
date_cols = ["basecamp_date", "highpoint_date", "termination_date"]
for col in date_cols:
    expeditions[col] = pd.to_datetime(expeditions[col])
expeditions[date_cols].dtypes

basecamp_date       datetime64[us]
highpoint_date      datetime64[us]
termination_date    datetime64[us]
dtype: object

### Unknown categories

`Unknown` in `season` and `termination_reason` really means the value is missing, so we change it to NaN.

In [7]:
expeditions["season"] = expeditions["season"].replace("Unknown", pd.NA)
expeditions["termination_reason"] = expeditions["termination_reason"].replace("Unknown", pd.NA)
print(expeditions["season"].isnull().sum(), expeditions["termination_reason"].isnull().sum())

2 96


### More than one country in one value

When a member has two citizenships (for example `Australia/UK`) we keep the first one. For first ascents (`UK, USA`) we also keep the first country.

In [8]:
members["citizenship"] = members["citizenship"].str.split("/").str[0].str.strip()
peaks["first_ascent_country"] = peaks["first_ascent_country"].str.split(",").str[0].str.strip()
print(members["citizenship"].str.contains("/", na=False).sum())

0


### Invalid values

A highpoint cannot be higher than the peak, and a 7 year old member is not realistic. These values are set to missing.

In [9]:
peak_height = peaks.set_index("peak_id")["height_metres"]
too_high = expeditions["highpoint_metres"] > expeditions["peak_id"].map(peak_height)
print(too_high.sum(), "highpoints above the peak height")
expeditions.loc[too_high, "highpoint_metres"] = pd.NA

print((members["age"] < 10).sum(), "member under 10 years old")
members.loc[members["age"] < 10, "age"] = pd.NA

49 highpoints above the peak height
1 member under 10 years old


### Missing values

Missing `age`, `highpoint_metres`, `trekking_agency` and dates are left as missing. We do not know the real values so filling them with an average would make the data wrong.

### Fix data types and remove unnecessary columns

The counts are stored as decimals, so we change them to whole numbers. `peak_name` is repeated in three files, so we keep it only in `peaks`. The columns with the height of death and injury are almost empty so we remove them.

In [10]:
peaks["first_ascent_year"] = peaks["first_ascent_year"].astype("Int64")
peaks["height_metres"] = peaks["height_metres"].astype("Int64")
expeditions["highpoint_metres"] = expeditions["highpoint_metres"].astype("Int64")
members["highpoint_metres"] = members["highpoint_metres"].astype("Int64")
members["age"] = members["age"].astype("Int64")

peaks = peaks.drop(columns=["peak_alternative_name", "first_ascent_expedition_id"])
expeditions = expeditions.drop(columns=["peak_name"])
members = members.drop(columns=["peak_id", "peak_name", "year", "season", "death_height_metres", "injury_height_metres"])

print(peaks.shape, expeditions.shape, members.shape)

(469, 6) (10364, 15) (76519, 15)


### Standardise text

Extra spaces are removed from all text columns so the same value is always written the same way.

In [11]:
for df in [peaks, expeditions, members]:
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].str.strip()
print("done")

done


## 3. Data transformation

### New columns for peaks, expeditions and members

- `height_category` for peaks
- `decade`, `success`, `total_deaths` and `duration_days` for expeditions
- `age_group` for members

An expedition is a success when `termination_reason` starts with Success (main peak, subpeak or claimed).

In [12]:
peaks["height_category"] = pd.cut(peaks["height_metres"], bins=[0, 5999, 7999, 9000],
                                  labels=["Under 6000 m", "6000-7999 m", "8000 m and above"])

expeditions["decade"] = (expeditions["year"] // 10) * 10
expeditions["success"] = expeditions["termination_reason"].str.startswith("Success").fillna(False)
expeditions["total_deaths"] = expeditions["member_deaths"] + expeditions["hired_staff_deaths"]
expeditions["duration_days"] = (expeditions["termination_date"] - expeditions["basecamp_date"]).dt.days

members["age_group"] = pd.cut(members["age"], bins=[0, 19, 29, 39, 49, 59, 100],
                              labels=["Under 20", "20-29", "30-39", "40-49", "50-59", "60 and above"])

print(expeditions["duration_days"].describe())

count    7946.000000
mean       26.566071
std        15.408913
min         0.000000
25%        14.000000
50%        25.000000
75%        37.000000
max       133.000000
Name: duration_days, dtype: float64


In [13]:
bad_duration = (expeditions["duration_days"] < 0) | (expeditions["duration_days"] > 365)
print(bad_duration.sum(), "wrong durations")
expeditions.loc[bad_duration, "duration_days"] = pd.NA
expeditions["duration_days"] = expeditions["duration_days"].astype("Int64")

0 wrong durations


### Check the transformed data

The deaths in `members` should match the deaths in `expeditions`.

In [14]:
print("Deaths in members:", int(members["died"].sum()))
print("Deaths in expeditions:", int(expeditions["member_deaths"].sum() + expeditions["hired_staff_deaths"].sum()))
print(expeditions["success"].value_counts())

Deaths in members: 1106
Deaths in expeditions: 1106
success
True     5727
False    4637
Name: count, dtype: int64


### Split into separate tables

The trekking agencies are moved into their own table (`agencies`). The expeditions table then uses `agency_id`.

In [15]:
agencies = pd.DataFrame({"agency_name": sorted(expeditions["trekking_agency"].dropna().unique())})
agencies["agency_id"] = agencies.index + 1
agencies = agencies[["agency_id", "agency_name"]]

agency_ids = dict(zip(agencies["agency_name"], agencies["agency_id"]))
expeditions["agency_id"] = expeditions["trekking_agency"].map(agency_ids).astype("Int64")
expeditions = expeditions.rename(columns={"members": "member_count"})

peaks = peaks[["peak_id", "peak_name", "height_metres", "height_category", "climbing_status",
               "first_ascent_year", "first_ascent_country"]]

expeditions = expeditions[["expedition_id", "peak_id", "agency_id", "year", "decade", "season",
                           "basecamp_date", "highpoint_date", "termination_date", "duration_days",
                           "termination_reason", "highpoint_metres", "member_count", "member_deaths",
                           "hired_staff", "hired_staff_deaths", "total_deaths", "oxygen_used", "success"]]

members = members[["member_id", "expedition_id", "sex", "age", "age_group", "citizenship",
                   "expedition_role", "hired", "success", "solo", "oxygen_used", "died",
                   "death_cause", "injured", "injury_type", "highpoint_metres"]]

for name, df in [("peaks", peaks), ("agencies", agencies), ("expeditions", expeditions), ("members", members)]:
    print(name, df.shape)

peaks (469, 7)
agencies (827, 2)
expeditions (10364, 19)
members (76519, 16)


## Saving the cleaned data

The four cleaned tables are saved as csv files so the next notebooks can use them.

In [16]:
peaks.to_csv("../data/processed/peaks.csv", index=False)
agencies.to_csv("../data/processed/agencies.csv", index=False)
expeditions.to_csv("../data/processed/expeditions.csv", index=False)
members.to_csv("../data/processed/members.csv", index=False)
print("saved")

saved
